# Visualização das bancas de defesa e qualificação do ICMC-USP

**SCC0252 – Visualização Computacional – 2026/2**
**Prof. Jean R. Ponciano**
ICMC-USP São Carlos

**Projeto de Exploração e Análise Visual de Dados – Entrega 1**

| Nome | Nº USP |
|------|--------|
| Gabriel da Costa Merlin | 12544420 |
| Guilherme Antonio Costa Bandeira | 14575620 |
| João Pedro Mori Machado | 13671831 |

## 0. Apresentação

### Motivação

As bancas de qualificação e de defesa do ICMC-USP são divulgadas por e-mail, em avisos semanais que informam aluno, orientador, título, data e local. Cada aviso mostra o que acontece naquela semana, mas ninguém enxerga o conjunto. Como alunos de graduação que pretendem ingressar na pós-graduação, temos pouca noção de como esse processo funciona na prática, e um conjunto de dados estruturado com visualizações interativas pode dar essa noção.

O interesse vai além do nosso:
- **Para quem pretende ingressar na pós ou se aproxima da própria banca:** saber quando as bancas costumam acontecer, como são as do programa de interesse (mestrado ou doutorado, presencial ou remota) e quais temas cada orientador acompanha ajuda a escolher programa e orientador e a planejar o cronograma.
- **Para o ICMC:** saber em que semanas as bancas se concentram e quantas são remotas ajuda a dimensionar salas, equipamentos de transmissão e apoio técnico.
- **Para os professores:** entender a carga de bancas em cada época e os temas que circulam em cada programa ajuda a se preparar para as bancas de seus orientandos e as de convidados.

### Contextualização do problema

O ICMC-USP São Carlos abriga cinco programas de pós-graduação: Ciências de Computação e Matemática Computacional (PPG-CCMC), Estatística (PIPGES), Matemática (PPG-Mat), Matemática, Estatística e Computação Aplicadas à Indústria (MECAI) e Matemática em Rede Nacional (PROFMAT). Em cada um, o aluno passa por bancas de qualificação e de defesa, no mestrado ou no doutorado, de forma presencial ou remota.

Essas informações existem, mas não estão organizadas para análise: chegam em e-mails de texto livre, e nenhuma tabela mostra como o volume de bancas varia ao longo do ano ou quais combinações de programa, nível, tipo e modalidade são comuns ou raras. O tema de cada orientador, em particular, não é um campo do aviso, e fica escondido no título do trabalho, que mistura português, inglês e vocabulário técnico de cada área.

Este trabalho transforma esses avisos em um conjunto de dados estruturado e o explora com visualizações interativas, em torno de três questões: quando as bancas acontecem, que perfis de banca existem e quais são raros, e se os orientadores se concentram em poucos temas ou cobrem áreas variadas. As seções seguintes descrevem o conjunto de dados, as questões e as visualizações.

## 1. Introdução

### 1.1 Conjunto de dados

#### Origem e coleta

O conjunto de dados foi construído a partir da agenda semanal de defesas e qualificações do ICMC-USP São Carlos, enviada por e-mail à lista `[grad-l]`. Cada e-mail informa, entre outros dados, a data, o nome do(a) aluno(a), o nome do(a) orientador(a), o título do trabalho e o local do evento ou o link de acesso. Um mesmo e-mail pode conter várias bancas, e a mesma banca pode aparecer em mais de um e-mail (agenda semanal, "Atualizado", "Retificação").

A coleta seguiu duas etapas:

1. **Exportação dos e-mails:** busca no Gmail da USP por `subject:"[grad-l]" "Qualificações"`, aplicação de um rótulo aos resultados e exportação desse rótulo em formato `.mbox` pelo Google Takeout. Foram exportados 240 e-mails (contagem do Gmail), enviados entre 11/03/2022 e 23/10/2026.
2. **Geração do CSV:** o script `data/scrape_emails.py` lê o `.mbox` e gera o `eventos.csv`, com uma banca por linha.

Foram lidas 1398 ocorrências de bancas, que resultaram em 1150 eventos únicos após a deduplicação (ver seção 2). As instruções para executar o projeto estão no `README.md`.

In [1]:
# Setup: imports das três questões, caminhos dos dados e configuração geral.
# As paletas acessíveis (daltonismo) são definidas junto do código de cada questão.

import html
import math
import unicodedata
import warnings
import zlib

import ipywidgets as widgets
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, clear_output, display
from plotly.subplots import make_subplots

# Caminhos relativos à pasta viz/, onde este notebook está
CAMINHO = "../data/eventos.csv"                    # dataset gerado por data/scrape_emails.py
ARQ_TEMAS = "question3/temas_llm_novo.csv"         # tema de cada banca, gerado por viz/question3/classificar_temas.py


#### Descrição do dataset

O dataset contém **1150 bancas** (defesas e qualificações de mestrado e doutorado) de **5 programas de pós-graduação** do ICMC, com **14 variáveis** por banca. Cada linha é uma banca. As bancas ocorrem entre 15/03/2022 e 29/10/2026, e os e-mails de origem foram enviados entre 11/03/2022 e 23/10/2026. São **930 alunos** e **162 orientadores** distintos.

Os programas são:
- Ciências de Computação e Matemática Computacional - PPG-CCMC (766 bancas)
- Estatística - PIPGES (165)
- Matemática - PPG-Mat (115)
- Matemática, Estatística e Computação Aplicadas à Indústria - MECAI (65)
- Matemática em Rede Nacional - PROFMAT (39)

**Dicionário de dados**

| Coluna | Descrição | Tipo de variável |
|---|---|---|
| `id` | Identificador da banca (hash de aluno + tipo + nível) | Identificador |
| `tipo` | Defesa ou Qualificação | Categórica nominal (binária) |
| `nivel` | Mestrado ou Doutorado | Categórica nominal (binária) |
| `programa` | Programa de pós-graduação, com a sigla no final | Categórica nominal (5 valores) |
| `titulo` | Título do trabalho | Texto livre |
| `aluno` | Nome do(a) aluno(a) | Texto (nominal) |
| `orientador` | Nome do(a) orientador(a), com grafia padronizada | Categórica nominal |
| `data` | Data da banca (AAAA-MM-DD) | Temporal |
| `dia_semana` | Dia da semana da banca (0 = segunda-feira, 4 = sexta-feira) | Categórica ordinal |
| `hora` | Horário da banca (HH:MM), vazio se o e-mail não informou | Temporal |
| `local` | Código da sala (se `icmc` = 1), texto do e-mail (se `icmc` = 0) ou vazio | Categórica nominal |
| `icmc` | 1 = sala do ICMC, 0 = outro lugar, vazio = sem local registrado (bancas remotas e uma presencial) | Categórica nominal (binária, com vazio) |
| `remoto` | 1 = remota, 0 = presencial | Categórica nominal (binária) |
| `atualizado` | 1 se a versão final veio de e-mail de correção | Categórica nominal (binária) |

**Particularidades dos dados**

O dataset é gerado por um script de extração sobre e-mails de texto livre, e não representa com 100% de fidelidade o conteúdo dos e-mails. As particularidades conhecidas são:

- **Cobertura temporal:** 2022 (a partir de março) e 2026 (até outubro) são anos incompletos, o que deve ser considerado ao comparar anos. Entre março de 2022 e outubro de 2026, todos os meses têm ao menos uma banca.
- **Horário:** `hora` está vazia em uma única banca (a qualificação de doutorado de Bruno Estanislau Holtz, em 13/08/2026), porque o e-mail não informou o horário.
- **Local das bancas remotas:** `local` e `icmc` ficam vazios nas bancas remotas (731 de 1150). Uma banca presencial também está sem local registrado (defesa de Misael Costa Junior, em 14/06/2024).
- **Locais fora do ICMC:** 16 bancas estão com `icmc` = 0, sendo 13 na UFSCar, 1 na FFCLRP (USP Ribeirão Preto), 1 defesa em sessão fechada (Maria Lydia Fioravanti, 29/05/2026), que não é remota nem tem sala, e 1 banca que foi classificada como fora do ICMC por um problema na extração: o local aparece como "Sala 3002 do ICMC-USP", sem hífen no código da sala.
- **Textos:** títulos e nomes seguem o que está no e-mail, então podem conter erros de digitação e cortes. Ao menos 3 títulos terminam de forma abrupta (em "de", em "com" e em vírgula).
- **Cancelamentos:** bancas canceladas sem aviso por e-mail não são detectadas.

### 1.2 Questões de investigação

**Q1. Quando as bancas acontecem?**
Como o volume de bancas varia ao longo do ano, e esse padrão se repete entre os anos e difere entre programas, tipos (defesa ou qualificação) e níveis (mestrado ou doutorado)?

**Variáveis usadas na Q1:** `data` (de onde saem a semana e o ano ISO), `programa`, `tipo` e `nivel` (filtros e painéis). `aluno` e `hora` aparecem apenas nas dicas (tooltips).

**Q2. Que perfis de banca existem e quais são raros?**
Que combinações de programa, nível, tipo e modalidade (remota ou presencial) dominam o dataset, e quais são raras?

**Q3. Os orientadores são especialistas ou generalistas?**
Os orientandos de cada professor se concentram em poucos temas ou cobrem uma gama de áreas?

> **TODO:** indicar as variáveis do dataset utilizadas na Q2 e na Q3.

In [2]:
# Análise preliminar que justifica as questões (poucas linhas por questão)

## 2. Pré-processamento

A limpeza e a transformação dos dados foram feitas durante a coleta, no script `data/scrape_emails.py`. O script lê o arquivo `defesas.mbox` exportado pelo Google Takeout e gera o `eventos.csv`, com um evento (banca) por linha. No notebook, a Q1 apenas deriva colunas auxiliares a partir de `data` e `programa`, sem gravá-las no CSV: `segunda` (segunda-feira da semana da banca), `sigla` (só a sigla do programa) e o ano e a semana ISO de cada semana. Além disso, a Q1 marca como "sem dados" sequências de 6 ou mais semanas sem nenhuma banca, para não confundi-las com zero. O processamento dos títulos usado na Q3 é apresentado na seção 3.3, junto às visualizações dessa questão.

**Extração**
- Só são lidos os e-mails cujo assunto contém "Defesas e Qualificações".
- Quando não há versão em texto puro, o HTML é convertido em texto. Os trechos citados de respostas anteriores ("Em ... escreveu:") são removidos.
- Cada e-mail é dividido em blocos, um por banca, e os campos são extraídos com expressões regulares: tipo (defesa ou qualificação), nível (mestrado ou doutorado), programa, título, aluno(a), orientador(a), data, hora e local.

**Limpeza e padronização**
- Tratamento de irregularidades de formatação dos e-mails: linhas quebradas no meio de nomes, títulos e programas, asteriscos de negrito, espaços duplicados, assinaturas e rodapés colados na agenda, e alguns erros de digitação pontuais.
- **Programas:** mapeados para cinco nomes padronizados, com a sigla no final (ex.: "Estatística - PIPGES").
- **Orientadores:** unificação das grafias de um mesmo nome (acentos, maiúsculas, ponto final e sobrenome omitido).
- **Ano da data:** os e-mails quase nunca informam o ano, então ele é inferido a partir da data de envio, com ajuste na virada de dezembro para janeiro.

**Deduplicação**
- A mesma banca pode aparecer em vários e-mails (agenda semanal, "Atualizado", "Retificação"). Cada banca recebe um `id` (hash de aluno + tipo + nível) e fica apenas a versão do e-mail mais recente.
- Quando uma atualização troca o nível da banca (ex.: Doutorado → Mestrado) com datas a até 90 dias de diferença, vale a versão mais nova.
- Foram lidas 1398 ocorrências de bancas nos e-mails, que resultaram em 1150 eventos únicos no CSV (248 ocorrências repetidas ou substituídas por versões mais recentes foram descartadas).

**Variáveis derivadas**
- `dia_semana`: dia da semana da data (0 = segunda-feira).
- `remoto`: 1 se o local indica banca remota, 0 caso contrário. Um caso avisado em texto livre ("passou a ser virtual") também é tratado.
- `icmc`: 1 se a banca é em sala do ICMC (e `local` passa a ter só o código da sala, ex.: 3-002), 0 se é em outro lugar, vazio se é remota.
- `atualizado`: 1 se o assunto do e-mail indica atualização ou retificação.

**Valores ausentes**
- Em bancas remotas, `local` e `icmc` ficam vazios por definição. Os demais campos que não puderem ser extraídos também ficam vazios.

### 3.1 Q1: Quando as bancas acontecem?

Como o volume de bancas varia ao longo do ano, e esse padrão se repete entre os anos e difere entre programas, tipos (defesa ou qualificação) e níveis (mestrado ou doutorado)?

**Justificativa das visualizações.** A Q1 pede duas comparações: como o volume de bancas varia ao longo do ano e se esse padrão se repete entre os anos. A variável de interesse é a contagem de bancas por semana, derivada de `data`, uma série temporal com ciclo anual. Por isso usamos um **gráfico sazonal de linhas (*cycle plot*)**: o eixo X é a semana do ano (ISO), o eixo Y é o número de bancas na semana, e há **uma linha por ano**. Como todos os anos ocupam o mesmo eixo, picos e vales que caem nas mesmas semanas indicam um padrão que se repete. Uma linha tracejada mostra a média dos anos, calculada só nas semanas com pelo menos dois anos de dados. A semana foi preferida ao mês porque preserva a variação dentro do mês.

Para a segunda parte da questão (o padrão difere entre programas, tipos e níveis?), a mesma visualização pode ser dividida em **painéis**, um por categoria, com as mesmas cores por ano e, opcionalmente, o mesmo eixo Y, o que permite comparar tanto o volume quanto a forma das curvas.

A interação permite filtrar por programa, tipo e nível, dividir em painéis, recortar o período, ligar ou desligar anos pela legenda e ver, na dica de cada ponto, as bancas daquela semana.

**Tratamento de dados ausentes.** Semanas sem e-mails não viram zero: a linha se interrompe e uma faixa cinza marca o intervalo. Semanas com e-mails, mas sem bancas da categoria filtrada, são zero de fato. Anos que não cobrem a semana 1 a 52 aparecem como "(parcial)" na legenda (2022 e 2026).

**Acessibilidade (daltonismo).** Os anos são distinguidos por dois canais, e não só pela cor. As cores seguem a paleta Okabe-Ito (Okabe & Ito, 2008), desenhada para as formas comuns de daltonismo, e cada ano tem também um símbolo de marcador próprio (círculo, quadrado, losango, triângulo e x). As linhas dos anos são sempre contínuas, de modo que a média dos anos, em preto e tracejada, é a única linha distinta por traço. 

In [3]:
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
CORES_ANO = ["#56B4E9", "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
SIMBOLOS_ANO = ["circle", "square", "diamond", "triangle-up", "x", "cross"]
CINZA = "#c4c8cf"
MAX_LINHAS_TOOLTIP = 10
_SEMANAS = list(range(1, 54))
_TICKS = [int(pd.Timestamp(2025, m, 1).isocalendar().week) for m in range(1, 13)]


def carregar(caminho_csv):
    df = pd.read_csv(caminho_csv, parse_dates=["data"]).dropna(subset=["data"]).copy()
    df["segunda"] = df["data"] - pd.to_timedelta(df["data"].dt.weekday, unit="D")
    df["sigla"] = df["programa"].str.rsplit(" - ", n=1).str[-1]
    return df


def montar_grade(df, gap_min=6, manual=()):
    """Uma linha por semana (segunda-feira) entre o 1º e o último evento.
    `gap` = sequência de >= gap_min semanas seguidas sem nenhuma banca (= sem e-mails, não zero).
    Recesso curto (dez/jan) continua zero. `manual`: [(inicio, fim), ...] força "sem dados".
    Calculado no df COMPLETO, antes dos filtros."""
    seg = pd.date_range(df["segunda"].min(), df["segunda"].max(), freq="7D")
    vazio = df.groupby("segunda").size().reindex(seg, fill_value=0).to_numpy() == 0
    gap = np.zeros(len(seg), bool)
    i = 0
    while i < len(seg):
        if vazio[i]:
            j = i
            while j < len(seg) and vazio[j]:
                j += 1
            if j - i >= gap_min:
                gap[i:j] = True
            i = j
        else:
            i += 1
    iso = seg.isocalendar()
    g = pd.DataFrame({"ano": iso["year"].astype(int).to_numpy(),
                      "semana": iso["week"].astype(int).to_numpy(), "gap": gap}, index=seg)
    for ini, fim in manual:
        g.loc[pd.Timestamp(ini):pd.Timestamp(fim), "gap"] = True
    return g

In [4]:
def _lista(d):
    if d is None or len(d) == 0:
        return ""
    d = d.sort_values(["data", "hora"], na_position="last")
    linhas = []
    for r in d.itertuples():
        h = f" {r.hora}" if isinstance(r.hora, str) else ""
        linhas.append(f"{r.data:%d/%m}{h} · {r.tipo}/{r.nivel} · {html.escape(str(r.aluno))} ({r.sigla})")
    extra = len(linhas) - MAX_LINHAS_TOOLTIP
    linhas = linhas[:MAX_LINHAS_TOOLTIP]
    if extra > 0:
        linhas.append(f"… e mais {extra}")
    return "<br>" + "<br>".join(linhas)


def _semanal(df, g, periodo=None):
    """Gera matrizes temporais de contagem e textos de tooltip de forma 100% segura.

    - Se a semana está num gap global do ano (sem e-mails), o valor é NaN (a linha quebra).
    - Se a semana está ativa, mas a categoria não possui eventos, o valor é exatamente 0.0."""
    anos = sorted(g["ano"].unique())
    ia = {a: k for k, a in enumerate(anos)}

    cont = np.full((len(anos), 53), np.nan)
    texto = np.full((len(anos), 53), "", dtype=object)

    ini, fim = periodo if periodo else (g.index[0], g.index[-1])
    grupos = dict(tuple(df.groupby("segunda")))

    for seg, r in g.iterrows():
        a_val = int(r["ano"])
        s_val = int(r["semana"])

        if a_val not in ia or s_val < 1 or s_val > 53:
            continue

        a_idx = ia[a_val]
        s_idx = s_val - 1

        # Verifica se a semana está fora dos limites de início/fim reais registrados para o ano
        limites_ano = g[g["ano"] == a_val].index
        fora_dos_limites = (seg < limites_ano.min()) or (seg > limites_ano.max())

        if r["gap"] or not (ini <= seg <= fim) or fora_dos_limites:
            cont[a_idx, s_idx] = np.nan
            texto[a_idx, s_idx] = f"<b>Semana {s_val}/{a_val}</b> ({seg:%d/%m})<br><i>sem dados de e-mail</i>"
        else:
            # Semana ativa com dados: se houver eventos, conta; se não, é estritamente 0.0
            d = grupos.get(seg)
            n = 0 if d is None else len(d)
            cont[a_idx, s_idx] = float(n)

            intervalo_texto = f"{seg:%d/%m}–{seg + pd.Timedelta(days=6):%d/%m}"
            texto[a_idx, s_idx] = f"<b>Semana {s_val}/{a_val}</b> ({intervalo_texto})<br><b>{n}</b> banca(s) real(is)" + _lista(d)

    return anos, cont, texto


def _suavizar(cont, k):
    """Calcula a média móvel sem deixar pontas flutuantes e respeitando NaNs."""
    if k <= 1:
        return cont
    out = np.full_like(cont, np.nan)
    for i, linha in enumerate(cont):
        s_linha = pd.Series(linha)
        # Roda a média móvel apenas onde temos valores válidos (incluindo os zeros que acabamos de fixar)
        m = s_linha.rolling(k, center=True, min_periods=1).mean().to_numpy()
        out[i] = np.where(np.isnan(linha), np.nan, m)
    return out


def _rotulo_ano(g, a):
    sem = g.loc[g["ano"] == a, "semana"]
    parcial = sem.min() > 1 or sem.max() < 52
    return f"{a} (parcial)" if parcial else str(a)


def _tracos(df, g, periodo, suavizar, media, mostrar_legenda, cor_ano):
    anos, cont, texto = _semanal(df, g, periodo)
    liso = _suavizar(cont, suavizar)
    tr = []
    for k, a in enumerate(anos):
        t = []
        for s_idx, (x, v_suave) in enumerate(zip(texto[k], liso[k])):
            if x == "" or "<i>" in x:
                t.append(x)
            else:
                v_real = cont[k, s_idx]
                if suavizar > 1 and not np.isnan(v_suave):
                    # Torna visualmente indubitável que a altura do ponto é a média móvel e não o valor bruto real
                    txt_suave = (
                        f"<br>-----------------------------"
                        f"<br><b>Valor real desta semana:</b> {int(v_real)} banca(s)"
                        f"<br><b>Média móvel (altura plotada):</b> {v_suave:.2f} banca(s)"
                        f"<br><i>*Suavização de {suavizar} semanas ativa</i>"
                    )
                    # Removemos a menção duplicada para evitar confusão
                    base_text = x.split("<br><b>")[0] + f"<br>({x.split(' (')[1].split(')')[0]})"
                    t.append(base_text + txt_suave)
                else:
                    t.append(x)

        tr.append(go.Scatter(
            x=_SEMANAS, y=liso[k], mode="lines+markers", name=_rotulo_ano(g, a),
            legendgroup=str(a), showlegend=mostrar_legenda,
            line=dict(color=cor_ano[a], width=2, shape="linear"),
            marker=dict(size=7, symbol=SIMBOLOS_ANO[k % len(SIMBOLOS_ANO)]), connectgaps=False,
            text=t, hovertemplate="%{text}<extra></extra>"))

    if media:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            m = np.nanmean(liso, axis=0)
        n = np.isfinite(liso).sum(axis=0)
        m = np.where(n >= 2, m, np.nan)
        tr.append(go.Scatter(
            x=_SEMANAS, y=m, mode="lines", name="média dos anos", legendgroup="media",
            showlegend=mostrar_legenda, line=dict(color="#222", width=2.5, dash="dash"),
            connectgaps=False,
            text=[f"<b>Semana {s}</b><br>média dos anos: {v:.1f} bancas<br>({k} anos com dados)"
                  for s, v, k in zip(_SEMANAS, m, n)],
            hovertemplate="%{text}<extra></extra>"))
    return tr


def _faixas_sem_dados(fig, g, row=None, col=None):
    for a in sorted(g["ano"].unique()):
        sub = g[(g["ano"] == a) & g["gap"]]
        if len(sub):
            x0 = int(sub["semana"].min())
            x1 = int(sub["semana"].max())
            fig.add_vrect(
                x0=x0 - 0.5, x1=x1 + 0.5, fillcolor=CINZA, opacity=0.15,
                layer="below", line_width=0, row=row, col=col
            )


def grafico(df, g, dividir=None, suavizar=1, periodo=None, media=True, y_comum=True, titulo=None):
    """dividir: None, 'programa', 'tipo' ou 'nivel' (um painel por categoria)."""
    cor_ano = {a: CORES_ANO[i % len(CORES_ANO)] for i, a in enumerate(sorted(g["ano"].unique()))}

    if not dividir:
        fig = go.Figure()
        tracos = _tracos(df, g, periodo, suavizar, media, True, cor_ano)
        for t in tracos:
            fig.add_trace(t)
        _faixas_sem_dados(fig, g)

        # Ajuste dinâmico de Y para painel único
        _, cont, _ = _semanal(df, g, periodo)
        liso = _suavizar(cont, suavizar)
        max_val = np.nanmax(liso) if np.any(np.isfinite(liso)) else 1
        teto_y = max(3, math.ceil(max_val * 1.15))

        fig.update_layout(
            title=titulo,
            xaxis=dict(tickmode="array", tickvals=_TICKS, ticktext=MESES, range=[0.5, 53.5], title="Semana do ano"),
            yaxis=dict(range=[0, teto_y], title="Nº de bancas na semana"),
            height=500, margin=dict(l=60, r=40, t=80, b=60), hovermode="closest",
            plot_bgcolor="#f8f9fa", paper_bgcolor="white"
        )
        return fig

    cats = sorted(df[dividir].dropna().unique())
    n_cats = len(cats)
    cols = 1 if n_cats <= 2 else 2
    rows = math.ceil(n_cats / cols)

    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=cats,
        shared_xaxes=True, shared_yaxes=y_comum,
        vertical_spacing=0.1, horizontal_spacing=0.08
    )

    # Calcular o valor máximo global para ajustar a escala do Y compartilhado ou individual
    max_global = 0.1
    for c in cats:
        sub_df = df[df[dividir] == c]
        _, cont, _ = _semanal(sub_df, g, periodo)
        liso = _suavizar(cont, suavizar)
        if np.any(np.isfinite(liso)):
            max_global = max(max_global, np.nanmax(liso))

    for idx, c in enumerate(cats):
        r_idx = (idx // cols) + 1
        c_idx = (idx % cols) + 1
        sub_df = df[df[dividir] == c]
        mostrar_leg = (idx == 0)
        tracos = _tracos(sub_df, g, periodo, suavizar, media, mostrar_leg, cor_ano)
        for t in tracos:
            fig.add_trace(t, row=r_idx, col=c_idx)
        _faixas_sem_dados(fig, g, row=r_idx, col=c_idx)

    # Determina o teto do eixo Y com folga de 15%
    teto_y = max(3, math.ceil(max_global * 1.15))

    fig.update_layout(
        title=titulo, height=330 * rows, margin=dict(l=60, r=40, t=80, b=60),
        hovermode="closest", plot_bgcolor="#f8f9fa", paper_bgcolor="white"
    )

    for r in range(1, rows + 1):
        for c in range(1, cols + 1):
            fig.update_xaxes(tickmode="array", tickvals=_TICKS, ticktext=MESES, range=[0.5, 53.5], row=r, col=c)
            fig.update_yaxes(range=[0, teto_y] if y_comum else [0, None], row=r, col=c)

    return fig


def painel_q1(caminho_csv, gap_min=6, manual=()):
    """Painel interativo sem opção de suavização. Requer ipywidgets."""
    W = widgets

    df = carregar(caminho_csv)
    g = montar_grade(df, gap_min, manual)

    def checks(titulo, opcoes):
        cbs = [W.Checkbox(value=True, description=str(o), indent=False,
                          layout=W.Layout(width="auto")) for o in opcoes]
        return W.VBox([W.HTML(f"<b>{titulo}</b>"), *cbs]), cbs

    box_p, cb_p = checks("Programa", df["sigla"].value_counts().index)
    box_t, cb_t = checks("Tipo", sorted(df["tipo"].unique()))
    box_n, cb_n = checks("Nível", sorted(df["nivel"].unique()))
    dividir = W.ToggleButtons(options=[("nenhum", None), ("programa", "programa"), ("tipo", "tipo"),
                                       ("nível", "nivel")], description="Dividir por")
    media = W.Checkbox(value=True, description="mostrar média dos anos", indent=False)
    ycom = W.Checkbox(value=True, description="mesmo eixo Y nos painéis", indent=False)
    slider = W.SelectionRangeSlider(
        options=[(f"{d:%d/%m/%Y}", d) for d in g.index], index=(0, len(g) - 1),
        description="Período", continuous_update=False,
        layout=W.Layout(width="95%"), style={"description_width": "60px"})

    # Criando container de saída e forçando a renderização do widget
    out = W.Output()

    def atualizar(*_):
        ps = [c.description for c in cb_p if c.value]
        ts = [c.description for c in cb_t if c.value]
        ns = [c.description for c in cb_n if c.value]
        ini, fim = slider.value
        d = df[df["sigla"].isin(ps) & df["tipo"].isin(ts) & df["nivel"].isin(ns)]
        n = d["segunda"].between(ini, fim).sum()
        # Chamar gráfico sem suavização (suavizar=1)
        fig = grafico(d, g, dividir.value, 1, (ini, fim), media.value, ycom.value,
                      titulo=f"Bancas por semana do ano, uma linha por ano · {n} bancas no filtro")
        with out:
            clear_output(wait=True)
            fig.update_layout(width=1100)
            display(fig)

    for w in cb_p + cb_t + cb_n + [slider, dividir, media, ycom]:
        w.observe(atualizar, "value")

    painel_completo = W.VBox([W.HBox([box_p, box_t, box_n, W.VBox([media, ycom])], layout=W.Layout(gap="40px")),
                              dividir, slider, out])

    display(painel_completo)
    atualizar()

#### Visualização 1.A: sazonalidade geral
Gráfico sazonal de linhas com todas as bancas: uma linha por ano e a média dos anos tracejada.

In [5]:
df_q1 = carregar(CAMINHO)
g_q1 = montar_grade(df_q1)

grafico(df_q1, g_q1, titulo="Bancas por semana do ano, uma linha por ano (todas as bancas)")

#### Visualização 1.B: painel interativo
Mesmo gráfico com filtros de programa, tipo e nível, divisão em painéis (por programa, tipo ou nível), recorte de período, média dos anos e eixo Y comum ou independente.

In [6]:
painel_q1(CAMINHO)

#### Leitura visual (Q1)

**Visão geral (1.A) e repetição entre os anos.** O padrão se repete entre os anos. Somando as bancas por mês em 2023, 2024 e 2025 (anos completos), os totais mensais de um ano acompanham os dos outros (correlação entre 0,73 e 0,80). Julho é o mês com menos bancas nos três anos (6 a 12) e outubro o mais movimentado (32 a 43). A repetição vem sobretudo do PPG-CCMC, que concentra 766 das 1150 bancas (67%). Nos programas menores, o padrão é menos estável: o PIPGES praticamente não se repete de um ano para o outro, e o PPG-Mat só em parte.

**Por programa (1.B, dividido por programa).**
- **PPG-CCMC:** é o programa com mais bancas, com média de 3,2 por semana, contra 0,2 a 0,7 nos demais, e é o único com padrão sazonal claro. Há alta em maio e junho, queda em julho e agosto, novo pico no fim de outubro e início de novembro e um pico em meados de dezembro. Na virada do ano a linha cai a zero, porque não há bancas nas últimas semanas de dezembro e em parte de janeiro. O maior volume semanal é de 13 bancas, em abril e maio de 2026 (semanas de 27/04, 04/05 e 25/05).
- **PIPGES (Estatística):** é o segundo maior programa. Na maioria das semanas tem 0 ou 1 banca, com picos de 4 a 6 bancas em agosto (o máximo, 6, na semana de 11/08/2025).
- **PPG-Mat:** na maioria das semanas tem 0 ou 1 banca, com mais bancas em fevereiro e março. O maior pico do programa, 8 bancas na semana de 13/02/2023, não se repetiu em nenhum outro ano.
- **MECAI e PROFMAT:** têm poucas bancas (65 e 39), com 0 ou 1 por semana na maior parte do tempo. Semanas com 2 bancas aparecem de vez em quando e, nesses programas, não configuram um pico. O máximo do MECAI é 3 bancas, na semana de 24/02/2025, e o do PROFMAT é 3 bancas, na semana de 07/11/2022.

**Por nível (1.B, dividido por nível).** Somando todos os programas, as bancas de doutorado variam menos ao longo do ano: ficam em geral entre 2 e 4 por semana e caem para cerca de 1 no meio do ano. As de mestrado oscilam mais: chegam a cerca de 6 por semana em maio e em outubro, caem para cerca de 1 nas férias de meio de ano e ficam em 3 ou 4 no restante.

**Por tipo (1.B, dividido por tipo).** As defesas são mais constantes ao longo do ano: ficam em cerca de 3 por semana na maior parte do tempo, com um pico no fim de fevereiro e início de março (média de 7 por semana nessa época) e outro em meados de dezembro (cerca de 6). Há uma queda para cerca de 1 em meados de julho. As qualificações são mais sazonais: começam o ano com 0 a 1 por semana e sobem de forma gradual até cerca de 5 no início de maio, caem para cerca de 1 em julho e agosto, voltam a subir até cerca de 6 em outubro e depois caem de novo. Ou seja, as qualificações se concentram em dois períodos do ano e as defesas se distribuem de forma mais uniforme. Esses valores são médias semanais entre os anos, e em semanas como a do pico de março a média depende de poucos anos (4 a 10 defesas por ano).

**Cuidados na leitura.**
- **Programas pequenos:** com poucas bancas por semana, as linhas oscilam entre 0, 1 e 2, e um único pico (como o do PPG-Mat em 2023) não indica padrão. Compare a forma das curvas só nos programas com mais bancas.
- **Anos parciais:** 2022 (a partir de março) e 2026 (até outubro) cobrem só parte do ano, então não podem ser comparados semana a semana com 2023 a 2025.
- **Eixo Y comum:** com o mesmo eixo nos painéis, as curvas dos programas pequenos ficam achatadas perto do zero, e o dominante é o CCMC.

### 3.2 Q2: Que perfis de banca existem e quais são raros?

Que combinações de programa, nível, tipo e modalidade (remota ou presencial) dominam o dataset, e quais são raras?

> **TODO:** justificar as visualizações escolhidas em função dos dados e da questão.

#### Visualização 2: treemap hierárquico
Treemap das defesas por programa, nível e modalidade, com a ordem da hierarquia escolhida pelos seletores. A área de cada caixa é proporcional ao número de defesas.

In [7]:
# Q2: constantes do treemap e carga das defesas

TIPO_DEFESA = 'defesa'
MAPA_MODALIDADE = {1: 'Remoto', 0: 'Presencial'}
COLUNAS_ANALISE = ['programa', 'nivel', 'modalidade']
SEPARADOR_PROGRAMA = ' - '   # o nome do programa é reduzido ao trecho depois do último separador

# Mapeamento amigável para os nomes dos seletores
OPCOES_DIMENSOES = {
    'Programa de Pós-Graduação': 'programa',
    'Nível (Mestrado/Doutorado)': 'nivel',
    'Modalidade (Remoto/Presencial)': 'modalidade',
}

# Paletas sequenciais, uma para cada categoria do nível externo: do branco até uma cor
# Okabe-Ito e depois uma versão escurecida dela. Cada programa tem a mesma cor das outras
# questões (CCMC azul, PIPGES laranja, PPG-Mat verde). As cinco cores continuam
# distinguíveis nas simulações de deuteranopia, protanopia e tritanopia.
def _escala(cor_hex):
    r, g, b = mcolors.to_rgb(cor_hex)
    return mcolors.LinearSegmentedColormap.from_list(cor_hex, [(1, 1, 1), (r, g, b), (0.45 * r, 0.45 * g, 0.45 * b)])

CMAPS_PROGRAMA = {
    'PPG-CCMC': _escala('#0072B2'),   # azul
    'PIPGES': _escala('#E69F00'),     # laranja
    'PPG-Mat': _escala('#009E73'),    # verde-azulado
    'MECAI': _escala('#CC79A7'),      # rosa
    'PROFMAT': _escala('#F0E442'),    # amarelo
}
# Quando o nível externo não é o programa (nível ou modalidade), as categorias usam as cores nesta ordem
CMAPS_DISPONIVEIS = list(CMAPS_PROGRAMA.values())

# Posição (0 a 1) em cada paleta
COR_EXTERNO = 0.4          # caixa do nível externo
COR_MEDIO = 0.55           # caixa do nível médio (um pouco mais escura)
COR_FOLHA_MIN = 0.3        # folha com a menor quantidade
COR_FOLHA_AMPLITUDE = 0.6  # quanto o tom sobe até a folha com a maior quantidade
COR_FOLHA_UNICA = 0.7      # quando todas as folhas têm a mesma quantidade

LARGURA, ALTURA = 1100, 650
TAMANHO_MIN_TEXTO = 10

df_q2 = pd.read_csv(CAMINHO)

# Filtrando apenas as Defesas (removendo qualificações para evitar duplicação)
df_q2_defesas = df_q2[df_q2['tipo'].str.lower() == TIPO_DEFESA].copy()

# Mapeando a coluna 'remoto' para termos descritivos legíveis
df_q2_defesas['modalidade'] = df_q2_defesas['remoto'].map(MAPA_MODALIDADE)


In [8]:
# Q2: funções do hover e do treemap

# Rótulos das dimensões, exibidos no hover (as que faltarem usam o nome da coluna)
ROTULOS_DIM = {
    'programa': 'Programa',
    'nivel': 'Nível',
    'modalidade': 'Modalidade',
}
MAX_IRMAOS = 8  # máximo de caixas irmãs com valor listadas no hover
MAX_ZEROS = 5   # máximo de caixas irmãs zeradas listadas no hover


def fmt_pct(x):
  return f'{100 * x:.1f}'.replace('.', ',') + '%'


def fmt_defesas(n):
  return f'{n} defesa' if n == 1 else f'{n} defesas'


def cor_texto(hex_cor):
  """Branco em fundo escuro, quase preto em fundo claro."""
  r, g, b = mcolors.to_rgb(hex_cor)
  return '#1a1a1a' if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else 'white'


def escurecer(hex_cor, fator=0.6):
  r, g, b = mcolors.to_rgb(hex_cor)
  return mcolors.to_hex((r * fator, g * fator, b * fator))


def montar_hover(path, dims, valor, valor_pai, total, irmaos):
  """Texto HTML do hover de uma caixa.

  path: rótulos do topo até a caixa (ex.: ['PPG-Mat', 'Mestrado', 'Remoto'])
  dims: nomes das dimensões desse mesmo caminho
  valor_pai: quantidade da caixa pai (None no nível externo)
  irmaos: {categoria: quantidade} de TODAS as categorias daquela dimensão,
          inclusive as que valem 0 (e por isso não têm retângulo)
  """
  atual = path[-1]
  linhas = [f'<b>{" › ".join(str(p) for p in path)}</b>']

  # Quantidade exata e porcentagens
  partes = [fmt_defesas(valor)]
  if valor_pai is not None:
    partes.append(f'{fmt_pct(valor / valor_pai)} do grupo')
  partes.append(f'{fmt_pct(valor / total)} do total')
  linhas.append(' · '.join(partes))

  # Posição entre as caixas irmãs e resumo delas (incluindo os zeros)
  if len(irmaos) > 1:
    posicao = 1 + sum(1 for v in irmaos.values() if v > valor)
    empate = sum(1 for v in irmaos.values() if v == valor) > 1
    rotulo = ROTULOS_DIM.get(dims[-1], dims[-1].capitalize())
    linhas.append(
        f'{rotulo} — {posicao}ª maior de {len(irmaos)}'
        + (' (empate)' if empate else '')
    )

    ordenados = sorted(irmaos.items(), key=lambda kv: (-kv[1], str(kv[0])))
    com_valor = [(c, v) for c, v in ordenados if v > 0]
    zeros = [str(c) for c, v in ordenados if v == 0]

    mostrados = com_valor[:MAX_IRMAOS]
    if all(c != atual for c, _ in mostrados):  # garante que a própria caixa apareça
      mostrados = mostrados[:-1] + [(atual, valor)]
    itens = [f'<b>{c}: {v}</b>' if c == atual else f'{c}: {v}' for c, v in mostrados]
    if len(com_valor) > len(mostrados):
      itens.append(f'… +{len(com_valor) - len(mostrados)}')
    linhas.append(' · '.join(itens))

    if zeros:
      txt = ', '.join(zeros[:MAX_ZEROS])
      if len(zeros) > MAX_ZEROS:
        txt += f' … +{len(zeros) - MAX_ZEROS}'
      linhas.append(f'Nenhuma defesa: {txt}')

  return '<br>'.join(linhas)


# Função que atualiza o gráfico dinamicamente com base nas escolhas do usuário
def atualizar_treemap(dim1, dim2, dim3):
  caminho = [dim1, dim2, dim3]

  # Validação para evitar dimensões repetidas na mesma hierarquia
  if len(set(caminho)) < 3:
    print(
        '⚠️ Aviso: Por favor, selecione três dimensões diferentes para cada'
        ' nível da hierarquia.'
    )
    return

  # Agrupamento base com as 3 dimensões
  df_raw = (
      df_q2_defesas.assign(
          programa=df_q2_defesas['programa'].str.rsplit(SEPARADOR_PROGRAMA, n=1).str[-1].str.strip()
      )
      .groupby([dim1, dim2, dim3])
      .size()
      .reset_index(name='quantidade')
  )

  # Totais e categorias usados no hover (porcentagens, posição e caixas zeradas)
  total_geral = df_raw['quantidade'].sum()
  categorias = {d: sorted(df_raw[d].unique()) for d in caminho}
  tot1 = df_raw.groupby(dim1)['quantidade'].sum().to_dict()
  tot2 = df_raw.groupby([dim1, dim2])['quantidade'].sum().to_dict()
  tot3 = df_raw.set_index([dim1, dim2, dim3])['quantidade'].to_dict()

  # Paleta de cada categoria do nível externo (dim1): por programa, ou pela ordem
  grupos_dim1 = df_raw[dim1].unique()
  mapa_cmaps = {
      g: CMAPS_PROGRAMA.get(g, CMAPS_DISPONIVEIS[i % len(CMAPS_DISPONIVEIS)])
      for i, g in enumerate(grupos_dim1)
  }

  # Listas para construir o DataFrame hierárquico explícito (pais, filhos e nós)
  ids, labels, parents, values, cores_hex, hovers = [], [], [], [], [], []

  # 1. Nível 1: Categoria mais externa (dim1)
  for g in grupos_dim1:
    sub_df = df_raw[df_raw[dim1] == g]
    val_g = sub_df['quantidade'].sum()

    id_1 = str(g)
    ids.append(id_1)
    labels.append(str(g))
    parents.append('')  # Raiz
    values.append(val_g)

    cmap = mapa_cmaps[g]
    # Cor base intermediária para a caixa externa
    cores_hex.append(mcolors.to_hex(cmap(COR_EXTERNO)))
    hovers.append(montar_hover(
        [g], [dim1], val_g, None, total_geral,
        {c: tot1.get(c, 0) for c in categorias[dim1]},
    ))

    # 2. Nível 2: Categoria intermediária (dim2)
    sub_grupos_dim2 = sub_df[dim2].unique()
    for sg2 in sub_grupos_dim2:
      sub2_df = sub_df[sub_df[dim2] == sg2]
      val_sg2 = sub2_df['quantidade'].sum()

      id_2 = f'{id_1} / {sg2}'
      ids.append(id_2)
      labels.append(str(sg2))
      parents.append(id_1)
      values.append(val_sg2)
      # Tom um pouco mais escuro para o nível intermediário
      cores_hex.append(mcolors.to_hex(cmap(COR_MEDIO)))
      hovers.append(montar_hover(
          [g, sg2], [dim1, dim2], val_sg2, val_g, total_geral,
          {c: tot2.get((g, c), 0) for c in categorias[dim2]},
      ))

      # 3. Nível 3: Folhas/Caixas mais internas (dim3) com gradiente baseado na quantidade
      min_val = sub_df['quantidade'].min()
      max_val = sub_df['quantidade'].max()

      for _, row in sub2_df.iterrows():
        sg3 = row[dim3]
        qtd = row['quantidade']

        id_3 = f'{id_2} / {sg3}'
        ids.append(id_3)
        labels.append(str(sg3))
        parents.append(id_2)
        values.append(qtd)

        # Gradiente interno proporcional à quantidade
        if max_val == min_val:
          norm_val = COR_FOLHA_UNICA
        else:
          norm_val = COR_FOLHA_MIN + COR_FOLHA_AMPLITUDE * (qtd - min_val) / (max_val - min_val)
        cores_hex.append(mcolors.to_hex(cmap(norm_val)))
        hovers.append(montar_hover(
            [g, sg2, sg3], caminho, qtd, val_sg2, total_geral,
            {c: tot3.get((g, sg2, c), 0) for c in categorias[dim3]},
        ))

  # Montando o DataFrame final com a estrutura completa da árvore
  df_tree = pd.DataFrame({
      'ids': ids,
      'labels': labels,
      'parents': parents,
      'values': values,
      'cor_hex': cores_hex,
      'hover': hovers,
      'cor_texto': [cor_texto(c) for c in cores_hex],
      'cor_borda': [escurecer(c) for c in cores_hex],
  })

  dicionario_cores = {c: c for c in df_tree['cor_hex'].unique()}

  # Criando o Treemap interativo
  fig = px.treemap(
      df_tree,
      ids='ids',
      names='labels',
      parents='parents',
      values='values',
      branchvalues='total',
      color='cor_hex',
      color_discrete_map=dicionario_cores,
      custom_data=['hover'],
      title=(
          '<b>Treemap Hierárquico Completo</b><br><sub>Hierarquia atual:'
          f' {" -> ".join(caminho)} | Cores customizadas por nível</sub>'
      ),
  )

  # Hover com fundo na cor da própria caixa; o <extra> remove a caixinha secundária
  fig.update_traces(
      hovertemplate='%{customdata[0]}<extra></extra>',
      hoverlabel=dict(
          bgcolor=df_tree['cor_hex'].tolist(),
          bordercolor=df_tree['cor_borda'].tolist(),
          font=dict(size=14, color=df_tree['cor_texto'].tolist()),
          align='left',
      ),
  )

  fig.update_layout(
      margin=dict(t=70, l=20, r=20, b=20),
      width=LARGURA,
      height=ALTURA,
      uniformtext=dict(minsize=TAMANHO_MIN_TEXTO, mode='hide'),
  )
  fig.show()


In [9]:
# Q2: painel interativo (ordem da hierarquia escolhida pelos seletores)

# Criando os dropdowns para o usuário escolher a ordem de aninhamento
q2_dropdown_nivel1 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='programa', description='1º (Externo):'
)
q2_dropdown_nivel2 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='nivel', description='2º (Médio):'
)
q2_dropdown_nivel3 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='modalidade', description='3º (Interno):'
)

# Organizando a interface visual com widgets do Jupyter
q2_ui = widgets.VBox(
    [
        widgets.HTML(
            '<b>Escolha a ordem de visualização (todas as caixas herdarão a'
            ' harmonia de cor do primeiro nível):</b>'
        ),
        widgets.HBox([q2_dropdown_nivel1, q2_dropdown_nivel2, q2_dropdown_nivel3]),
    ]
)

# Conectando os seletores à função de atualização
q2_out = widgets.interactive_output(
    atualizar_treemap,
    {'dim1': q2_dropdown_nivel1, 'dim2': q2_dropdown_nivel2, 'dim3': q2_dropdown_nivel3},
)

display(q2_ui, q2_out)


Output()

#### Leitura visual (Q2)

> **TODO:** descrever perfis dominantes e raros.

### 3.3 Q3: Os orientadores são especialistas ou generalistas?

Os orientandos de cada professor se concentram em poucos temas ou cobrem uma gama de áreas?

> **TODO:** justificar as visualizações escolhidas em função dos dados e da questão.

In [10]:
# Q3: constantes e junção das bancas com o tema de cada uma

ESPACO = HTML('<div style="height:48px"></div>')      # espaço entre os controles e o gráfico
ROTULO = {"tema": "Tema", "macro_tema": "Macro-tema"}      # nome de cada nível nos controles, hovers e títulos
COR = {"PPG-CCMC": "#0072B2", "PIPGES": "#E69F00", "PPG-Mat": "#009E73"}   # paleta que funciona com daltonismo

eventos = pd.read_csv(CAMINHO)[["id", "orientador"]]
temas = pd.read_csv(ARQ_TEMAS)
bancas = temas.merge(eventos, on="id")
n_outros = (bancas.tema == "Outros").sum()
bancas = bancas[bancas.tema != "Outros"]

PROGRAMAS = [p for p in COR if p in set(bancas.programa)]     # na ordem da paleta
MAX_BANCAS = int(bancas.orientador.value_counts().max())      # teto do slider "Mín. bancas": o orientador com mais bancas
print(f"{len(bancas)} bancas de {bancas.orientador.nunique()} orientadores ({n_outros} bancas 'Outros' ficaram de fora)")
bancas[["orientador", "programa", "tema", "macro_tema"]].head()


1005 bancas de 136 orientadores (41 bancas 'Outros' ficaram de fora)


,orientador,programa,tema,macro_tema
0,Renata Pontin de Mattos Fortes,PPG-CCMC,Engenharia de Software e Sistemas de Informação,Engenharia de Software e Sistemas de Informaçã...
1,Vicente Garibay Cancho,PIPGES,Modelos de Regressão,Modelos de Regressão
2,Jorge Luis Bazán Guzmán,PIPGES,Modelos de Regressão,Modelos de Regressão
3,André Carlos Ponce de Leon Ferreira de Carvalho,PPG-CCMC,Aprendizado de Máquina e Mineração de Dados,Inteligência Computacional
4,João Luis Garcia Rosa,PPG-CCMC,Sistemas Embarcados e Evolutivos,Sistemas Distribuídos e Programação Concorrent...


In [11]:
# Q3: medida de concentração por orientador e painel de controles

def sem_acento(s):
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()

def casa(nome, busca):
    return busca.strip() != "" and sem_acento(busca.strip()) in sem_acento(nome)

def curto(s, n=22):
    return s if len(s) <= n else s[: n - 1] + "…"

def perfis(programa, minimo, nivel):
    """Uma linha por orientador (perfil) e a tabela orientador x tema com o nº de bancas (conta)."""
    b = bancas if programa == "Todos" else bancas[bancas.programa == programa]
    conta = pd.crosstab(b.orientador, b[nivel])
    n = conta.sum(axis=1)
    conta, n = conta[n >= minimo], n[n >= minimo]
    p = conta.div(n, axis=0)
    perfil = pd.DataFrame({
        "n": n,
        "tema_principal": conta.idxmax(axis=1),
        "concentracao": p.max(axis=1),
    })
    perfil["programa"] = b.groupby("orientador").programa.agg(lambda s: s.value_counts().index[0]).reindex(perfil.index)
    return perfil, conta


def painel_q3(grafico, **controles):
    """Controles + gráfico numa caixa só. Parecido com o interact, mas a cada mudança a saída é
    limpa NA HORA antes de redesenhar (o interact limpa só quando chega a próxima saída)."""
    saida = widgets.Output(layout=widgets.Layout(width="100%"))     # ocupa a largura toda da saída

    def redesenhar(*_):
        saida.clear_output()
        with saida:
            grafico(**{nome: c.value for nome, c in controles.items()})

    for c in controles.values():
        c.observe(redesenhar, names="value")
    display(widgets.VBox([*controles.values(), saida], layout=widgets.Layout(width="100%")))
    redesenhar()


#### Visualização 3.A: dispersão por orientador
Um ponto por orientador: número de bancas no eixo horizontal e concentração no tema principal no vertical, com cor por programa. Filtros de programa, nível (tema ou macro-tema), mínimo de bancas e busca por nome.

In [12]:
def grafico_dispersao(programa="Todos", nivel="tema", minimo=4, busca=""):
    rot = ROTULO[nivel]
    perfil, _ = perfis(programa, minimo, nivel)
    perfil = perfil.assign(x=[n + ((zlib.crc32(o.encode()) % 1000) / 1000 - 0.5) * 0.36
                              for o, n in zip(perfil.index, perfil.n)])
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    fig = go.Figure()
    for prog in PROGRAMAS:
        q = perfil[perfil.programa == prog]
        if q.empty:
            continue
        texto = [f"<b>{o}</b> ({r.n} bancas)<br>{rot} principal: {r.tema_principal}"
                 f"<br>Concentração no {rot.lower()} principal: {r.concentracao:.0%} ({round(r.concentracao * r.n)}/{r.n})" for o, r in q.iterrows()]
        fig.add_trace(go.Scatter(x=q.x, y=q.concentracao, mode="markers", name=prog, text=texto, hoverinfo="text",
                                 marker=dict(color=COR[prog], size=10, opacity=0.85, line=dict(color="white", width=1))))

    a, b = (perfil.n.min(), perfil.n.max()) if len(perfil) else (1, MAX_BANCAS)

    achados = perfil[[casa(o, busca) for o in perfil.index]]
    if busca.strip():
        print(f"{len(achados)} encontrado(s) (circulado)" if len(achados) else "Nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    if len(achados):
        fig.add_trace(go.Scatter(x=achados.x, y=achados.concentracao, mode="markers+text", text=list(achados.index),
                                 textposition="top center", showlegend=False, hoverinfo="skip",
                                 marker=dict(size=20, color="rgba(0,0,0,0)", line=dict(color="black", width=2))))

    fig.update_xaxes(title_text="Nº de bancas do orientador", title_font=dict(size=12), range=[a - 0.7, b + 0.7], dtick=1, zeroline=False)
    fig.update_yaxes(title_text=f"Concentração no {rot.lower()} principal (% das bancas)", title_font=dict(size=12), range=[0, 1.04], tickformat=".0%",
                     zeroline=False)
    fig.update_layout(height=470, margin=dict(l=70, r=20, t=10, b=120), plot_bgcolor="#fafafa",
                      legend=dict(orientation="h", y=-0.28, x=0))
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída; sem a barra do plotly


painel_q3(grafico_dispersao,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa"),
       nivel=widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível"),
       minimo=widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas"),
       busca=widgets.Text(placeholder="Digite um nome", description="Buscar"))


#### Visualização 3.B: mapa de calor orientador × tema
Cada coluna é um orientador e cada linha um tema, com a cor indicando a fração das bancas do orientador naquele tema (cinza = nenhuma banca). Mesmos filtros da 3.A, mais a ordem das colunas.

In [13]:
ESCALA = [[0, "#e5e7eb"], [0.00001, "#e5e7eb"], [0.0000101, "#c6dbef"], [0.25, "#6baed6"],
          [0.5, "#3182bd"], [0.75, "#08519c"], [1, "#08306b"]]       # zero = cinza; resto, azul claro -> escuro

def grafico_heatmap(programa="Todos", nivel="tema", minimo=4, ordem="esp", busca=""):
    rot = ROTULO[nivel]
    perfil, conta = perfis(programa, minimo, nivel)
    perfil = {
        "esp": perfil.sort_values(["concentracao", "n"], ascending=False),
        "ger": perfil.sort_values(["concentracao", "n"], ascending=[True, False]),
        "n": perfil.sort_values("n", ascending=False),
        "nome": perfil.sort_index(),
    }[ordem]
    conta = conta.loc[perfil.index]
    conta = conta.loc[:, conta.sum() > 0]                     # só as colunas (temas) que alguém usou
    pct = conta.div(perfil.n, axis=0)
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    z = pct.values.T                                                   # linhas = temas, colunas = orientadores
    texto = [[f"<b>{o}</b> ({perfil.n[o]} bancas)<br>{rot}: {t}<br>" + (f"Concentração: {pct.loc[o, t]:.0%} das bancas ({conta.loc[o, t]}/{perfil.n[o]})"
                                                                      if conta.loc[o, t] else "Nenhuma banca nesse tema")
              for o in conta.index] for t in conta.columns]
    fig = go.Figure(go.Heatmap(x=list(conta.index), y=list(conta.columns), z=z,
                               text=texto, hoverinfo="text", colorscale=ESCALA, zmin=0,
                               zmax=1, xgap=0, ygap=0,
                               colorbar=dict(title="% das bancas",
                                             tickformat=".0%",
                                             x=1.01, xanchor="left", y=1, yanchor="top",
                                             lenmode="pixels", len=260, thickness=14)))
    idx = [i for i, o in enumerate(conta.index) if casa(o, busca)]
    if busca.strip():
        print(f"{len(idx)} encontrado(s) (coluna com contorno preto)" if idx else "Nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    for i in idx:
        fig.add_shape(type="rect", xref="x", yref="y", x0=i - 0.5, x1=i + 0.5, y0=-0.5, y1=len(conta.columns) - 0.5,
                      line=dict(color="black", width=2.5), layer="above")
    topo = int(0.82 * 5.5 * max((len(o) for o in conta.index), default=0)) + 40      # altura do maior nome girado em 55 graus
    fig.update_xaxes(tickangle=-55, side="top", automargin=True, tickfont=dict(size=9), type="category", fixedrange=True)
    fig.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=9), fixedrange=True, tickvals=list(conta.columns),
                     ticktext=[curto(t) for t in conta.columns])
    fig.update_layout(height=max(380, topo + 26 * len(conta.columns) + 40), margin=dict(l=130, r=20, t=topo, b=40), dragmode=False,
                      plot_bgcolor="white")
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída


painel_q3(grafico_heatmap,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa"),
       nivel=widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível"),
       minimo=widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas"),
       ordem=widgets.Dropdown(options=[("Mais especialistas primeiro", "esp"), ("Mais generalistas primeiro", "ger"),
                                       ("Nº de bancas", "n"), ("Nome", "nome")], description="Ordem"),
       busca=widgets.Text(placeholder="Digite um nome", description="Buscar"))


#### Leitura visual (Q3)

> **TODO:** descrever exemplos de especialistas e generalistas e cuidados na interpretação.

## 4. Discussões Preliminares

### Insights por questão

> **TODO:** responder Q1, Q2 e Q3 com base nas evidências visuais.

### Interpretação crítica e limitações

> **TODO:** discutir limitações dos dados e das visualizações.

### Reflexão e lições aprendidas

> **TODO:** descrever o processo e as lições aprendidas.

## 5. Próximos Passos

> **TODO:** descrever as direções para a segunda entrega (novas questões ou aprofundamento de alguma questão específica).

## 6. Referências e Anexos

> **TODO:** listar bibliotecas utilizadas, fontes dos dados e declaração de uso de ferramentas.